In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataStorageDemo").getOrCreate()

In [0]:
df_csv= spark.read.csv("/databricks-datasets/airlines/part-00000",inferSchema=True,header=True)


In [0]:
df_csv.printSchema()


In [0]:
display(df_csv.limit(5))

In [0]:
# /tmp usually maps to Instance Store rather than
csv_path = "/Volumes/workspace/default/files/airlines_csv"
#parquet_path = "/tmp/airlines_parquet"
parquet_path = "/Volumes/workspace/default/files/airlines_parquet"
#delta_path = "/tmp/airlines_delta"
delta_path = "/Volumes/workspace/default/files/airlines_delta"

df_csv.write.mode("overwrite").csv(csv_path)
df_csv.write.mode("overwrite").parquet(parquet_path)
df_csv.write.mode("overwrite").format("delta").save(delta_path)


In [0]:
df_csv= spark.read.csv(csv_path)
df_parquet = spark.read.parquet(parquet_path)
df_delta = spark.read.format("delta").load(delta_path)

print("CSV Count:", df_csv.count())          # reads line-by-line.
print("Parquet Count:", df_parquet.count())  # reads structured data in columnar format.
print("Delta Count:", df_delta.count())      # delta can count from metadata in logs like _delta_logs/00000000000000000000.json



In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, delta_path)

# Update cancelled field where destination is 'LAX'
deltaTable.update(
    condition="Dest = 'LAX'",
    set={"Cancelled": "'1'"}
)

# Read previous version
df_old = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
display(df_old.filter("Dest = 'LAX'").limit(4))

df_new = spark.read.format("delta").load(delta_path)
display(df_new.filter("Dest = 'LAX'").limit(4))
